# S3 - Baseline e curva de aprendizado

Este notebook publica a evidência de desenvolvimento já executada pelo
módulo S3. A fronteira é explícita: `train` é a partição de ajuste,
`validation` é a partição de avaliação e seleção, e teste, stress e monitor
permanecem selados. Grupos ambíguos ficam fora da avaliação científica.

A curva científica é praticamente plana: mais volume não melhora
`debt_credit_management`. O baseline supera o dummy, mas a visão
operacional all-text é reportada separadamente. Este resultado é validação
de desenvolvimento, não é confirmatório e não está pronto para deploy.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'dataset' / 'processed' / 'complaints.parquet').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not find project root')
    PROJECT_ROOT = parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.config import ProjectPaths
from consumer_complaint_intelligence.s3 import FrozenS3Protocol
from consumer_complaint_intelligence.s3 import validate_frozen_report

paths = ProjectPaths.from_root(PROJECT_ROOT)
protocol = FrozenS3Protocol.from_json(PROJECT_ROOT / 'config' / 's3_protocol.json')
report = paths.temp_dir / 's2' / 's2_report_pilot.json'
validate_frozen_report(protocol, report)
print({'protocol': protocol.protocol_id, 'status': protocol.approval_status})


{'protocol': 'post_2023_taxonomy', 'status': 'FROZEN_FOR_S3_DEVELOPMENT'}


In [2]:
from consumer_complaint_intelligence.s3 import BaselineConfig
from consumer_complaint_intelligence.s3 import build_or_load_development_dataset
from consumer_complaint_intelligence.s3 import run_s3_full
from consumer_complaint_intelligence.s3 import run_s3_smoke
from consumer_complaint_intelligence.s3_reporting import load_s3_evidence_tables

RUN_MODE = 'disabled'
if RUN_MODE not in {'disabled', 'smoke', 'full'}:
    raise ValueError("RUN_MODE must be 'disabled', 'smoke', or 'full'")

baseline_config = BaselineConfig(
    max_features=40000,
    min_df=2,
    max_df=0.98,
    ngram_range=(1, 2),
    max_iter=500,
    random_state=42,
    fractions=(0.25, 0.5, 1.0),
)
development_path = paths.temp_dir / 's3' / 'development.parquet'
full_artifact = paths.temp_dir / 's3' / 's3_full.json'


if RUN_MODE == 'disabled':
    print('S3 disabled; no corpus or model was executed.')
    if full_artifact.exists():
        evidence = load_s3_evidence_tables(full_artifact)
        display(evidence.curve)
        display(evidence.per_class)
else:
    cache = build_or_load_development_dataset(
        paths.parquet_path,
        paths.temp_dir / 's2' / 'modeling_index.parquet',
        development_path,
        protocol,
        temp_directory=paths.temp_dir / 'duckdb',
    )
    if RUN_MODE == 'smoke':
        result = run_s3_smoke(
            development_path, paths.temp_dir / 's3' / 's3_smoke.json'
        )
    else:
        result = run_s3_full(
            development_path,
            full_artifact,
            paths.temp_dir / 's3' / 'scientific.parquet',
            config=baseline_config,
            batch_size=4096,
            memory_limit='4GB',
            threads=1,
        )
    print({'mode': RUN_MODE, 'cache': cache['status'],
           'points': list(result['points'])})
    if RUN_MODE == 'full':
        evidence = load_s3_evidence_tables(full_artifact)
        display(evidence.curve)
        display(evidence.per_class)


S3 disabled; no corpus or model was executed.


fraction,train_groups,macro_f1,balanced_accuracy,debt_credit_management_f1,operational_macro_f1
f64,i64,f64,f64,f64,f64
0.25,86392,0.701704,0.727955,0.271543,null
0.5,172778,0.699611,0.73503,0.248733,null
1.0,345552,0.700393,0.741993,0.24581,0.678421


product_family,precision,recall,f1,support
str,f64,f64,f64,i64
"""cards_prepaid""",0.717413,0.735679,0.726431,17282
"""consumer_lending""",0.559582,0.690957,0.618369,6436
"""credit_reporting""",0.941057,0.911724,0.926158,164801
"""debt_collection""",0.693733,0.675064,0.684271,25565
"""debt_credit_management""",0.173873,0.419283,0.24581,892
"""deposit_accounts""",0.754761,0.813751,0.783147,15635
"""money_services""",0.658164,0.67589,0.666909,5421
"""mortgage""",0.788688,0.888853,0.83578,6181
"""student_loan""",0.77205,0.866737,0.816658,3767
